In [1]:
# =====================================================
# exp_wgt_return_6m 回测：15 市值组 + 指定组别内因子低分位等权
#
# 与原模拟策略保持一致的部分：
# - 市值分组、因子计算、行业过滤、趋势仓位与 20 日调仓规则
# - T 日收盘后形成信号，T+1 日开盘执行
# - 开盘涨跌停、停牌与成交量约束
#
# 本回测版仅作两项调整：
# 1) 初始资金改为 100,000 元；
# 2) 候选池在市值分组前剔除科创板（688/689）与创业板（300/301）。
# =====================================================

import math
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import dai
from bigquant import bigtrader


# =====================================================
# 1. 策略参数区
# =====================================================

# 回测起点同时决定固定的 20 日调仓相位；如需和模拟策略对照，请不要改变。
START_DATE = "2023-01-01"

# 回测终点。默认取运行当天；若需要固定样本期，可改为如 "2026-09-11"。
END_DATE = datetime.today().strftime("%Y-%m-%d")

MCAP_GROUP_COUNT = 15
SELECT_MCAP_GROUPS = [1]
GROUP_SELECT_PCT = 0.10
REBALANCE_DAYS = 20

TARGET_TOTAL_WEIGHT = 0.98
DEFENSIVE_TOTAL_WEIGHT = 0.30
TREND_INDEX = "000852.SH"
TREND_MA_DAYS = 30

FACTOR_MONTHS = 6
LOOKBACK_DAYS = 21 * FACTOR_MONTHS
BEFORE_START_DAYS = max(300, TREND_MA_DAYS * 3)
MIN_LIST_DAYS = int(LOOKBACK_DAYS * 365 / 252) + 30

# 本次回测初始资金：10 万元。
CAPITAL_BASE = 100_000

EXCLUDE_INDUSTRIES = [
    "交通运输",
    "电力及公用事业",
    "纺织服装",
    "轻工制造",
    "家电",
    "石油石化",
    "综合",
    "银行",
]

BENCHMARK = "000300.SH"
VERBOSE = False
BUY_COST = 0.0003
SELL_COST = 0.0013
# 不设单笔最低佣金：按买入/卖出费率实际计费。
MIN_COST = 0.0
USE_CURRENT_BAR_MATCHING = True
EPS = 1e-10


# =====================================================
# 2. 参数与通用辅助函数
# =====================================================

def _normalize_mcap_groups(groups, group_count):
    if groups is None or len(groups) == 0:
        raise ValueError("SELECT_MCAP_GROUPS 不能为空，例如 [1, 2, 3]。")

    normalized = sorted({int(g) for g in groups})
    invalid = [g for g in normalized if g < 1 or g > group_count]
    if invalid:
        raise ValueError(
            f"SELECT_MCAP_GROUPS 中存在非法组别 {invalid}；"
            f"有效范围为 1 到 {group_count}。"
        )
    return normalized


def _sql_quote(values):
    if values is None or len(values) == 0:
        raise ValueError("SQL 列表不能为空。")
    return ", ".join("'" + str(x).replace("'", "''") + "'" for x in values)


def _sql_int_list(values):
    if values is None or len(values) == 0:
        raise ValueError("SQL 整数列表不能为空。")
    return ", ".join(str(int(x)) for x in values)


def _safe_float(value, default=np.nan):
    try:
        if value is None or pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def _validate_parameters():
    global SELECT_MCAP_GROUPS
    SELECT_MCAP_GROUPS = _normalize_mcap_groups(
        SELECT_MCAP_GROUPS, MCAP_GROUP_COUNT
    )

    if not 0 < GROUP_SELECT_PCT <= 1:
        raise ValueError("GROUP_SELECT_PCT 必须位于 (0, 1] 区间内。")
    if REBALANCE_DAYS <= 0:
        raise ValueError("REBALANCE_DAYS 必须为正整数。")
    if not 0 <= DEFENSIVE_TOTAL_WEIGHT <= TARGET_TOTAL_WEIGHT <= 1:
        raise ValueError("仓位参数必须满足 0 <= DEFENSIVE <= TARGET <= 1。")


def _lag(field, periods):
    return field if periods == 0 else f"m_lag({field}, {periods})"


def _build_exp_wgt_return_expr():
    """构造与原策略相同的 6 个月成交量加权衰减收益表达式。"""
    numerator_terms = []
    denominator_terms = []

    for i in range(LOOKBACK_DAYS):
        decay_weight = math.exp(-i / FACTOR_MONTHS / 4.0)
        close_i = _lag("close", i)
        close_i_1 = _lag("close", i + 1)
        turn_i = _lag("turn", i)
        daily_return = f"(({close_i} / NULLIF({close_i_1}, 0)) - 1.0)"

        numerator_terms.append(
            f"COALESCE(({decay_weight:.12g} * {turn_i} * {daily_return}), 0.0)"
        )
        denominator_terms.append(
            f"COALESCE(({decay_weight:.12g} * {turn_i}), 0.0)"
        )

    return " + ".join(numerator_terms), " + ".join(denominator_terms)


# =====================================================
# 3. 信号构造
# =====================================================

def build_signal_data():
    _validate_parameters()

    calc_start_date = (
        datetime.strptime(START_DATE, "%Y-%m-%d")
        - timedelta(days=BEFORE_START_DAYS)
    ).strftime("%Y-%m-%d")

    industry_sql = _sql_quote(EXCLUDE_INDUSTRIES)
    selected_group_sql = _sql_int_list(SELECT_MCAP_GROUPS)
    numerator_expr, denominator_expr = _build_exp_wgt_return_expr()

    # 每天的趋势结果都只使用当日收盘前已有数据；在下一交易日才会生效。
    trend_sql = f"""
    WITH index_raw AS (
        SELECT
            date,
            close AS trend_index_close,
            AVG(close) OVER (
                PARTITION BY instrument
                ORDER BY date
                ROWS BETWEEN {TREND_MA_DAYS - 1} PRECEDING AND CURRENT ROW
            ) AS trend_index_ma
        FROM cn_stock_index_bar1d
        WHERE instrument = '{TREND_INDEX}'
          AND date >= '{calc_start_date}'
          AND date <= '{END_DATE}'
    )
    SELECT
        date,
        trend_index_close,
        trend_index_ma,
        CASE
            WHEN trend_index_ma IS NULL THEN 0
            WHEN trend_index_close > trend_index_ma THEN 1
            ELSE 0
        END AS risk_on,
        CASE
            WHEN trend_index_ma IS NULL THEN {DEFENSIVE_TOTAL_WEIGHT}
            WHEN trend_index_close > trend_index_ma THEN {TARGET_TOTAL_WEIGHT}
            ELSE {DEFENSIVE_TOTAL_WEIGHT}
        END AS trend_total_weight
    FROM index_raw
    WHERE date >= '{START_DATE}'
    ORDER BY date ASC
    """

    daily_trend_raw = dai.query(trend_sql).df()
    if daily_trend_raw.empty:
        raise ValueError("趋势指数数据为空，请检查 TREND_INDEX 与日期区间。")

    daily_trend_raw["date"] = (
        pd.to_datetime(daily_trend_raw["date"]).dt.strftime("%Y-%m-%d")
    )
    daily_trend_raw = daily_trend_raw.sort_values("date").reset_index(drop=True)
    all_dates = daily_trend_raw["date"].tolist()
    if len(all_dates) < 2:
        raise ValueError("交易日数量不足，无法形成 T 日信号、T+1 日执行。")

    # 交易日 t 只能使用 t-1 的收盘趋势信号。
    daily_trend_trade = daily_trend_raw.copy()
    daily_trend_trade["trend_signal_date"] = daily_trend_trade["date"].shift(1)
    for column in [
        "trend_index_close",
        "trend_index_ma",
        "risk_on",
        "trend_total_weight",
    ]:
        daily_trend_trade[column] = daily_trend_trade[column].shift(1)

    daily_trend_trade = daily_trend_trade.dropna(
        subset=["trend_signal_date", "trend_total_weight"]
    ).copy()
    daily_trend_trade["risk_on"] = daily_trend_trade["risk_on"].astype(np.int8)
    daily_trend_trade["trend_total_weight"] = daily_trend_trade[
        "trend_total_weight"
    ].astype(np.float32)

    # 调仓信号由固定相位产生，且在下一交易日执行。
    rebalance_signal_dates = all_dates[::REBALANCE_DAYS]
    next_date_map = {
        all_dates[i]: all_dates[i + 1] for i in range(len(all_dates) - 1)
    }
    rebalance_signal_dates = [
        date for date in rebalance_signal_dates if date in next_date_map
    ]
    if not rebalance_signal_dates:
        raise ValueError("调仓日期为空，请检查 START_DATE 或 REBALANCE_DAYS。")

    rebalance_date_sql = _sql_quote(rebalance_signal_dates)
    rebalance_schedule_df = pd.DataFrame(
        {
            "rebalance_signal_date": rebalance_signal_dates,
            "effective_start_date": [
                next_date_map[date] for date in rebalance_signal_dates
            ],
        }
    )

    # 排除规则通过证券代码前缀实现：
    # 科创板为 688/689 开头的沪市股票，创业板为 300/301 开头的深市股票。
    # 放在 factor_base 中，确保它们不参与市值分位和后续排序。
    signal_sql = f"""
    WITH factor_base AS (
        SELECT
            date,
            instrument,
            cs_level1_name,
            total_market_cap,
            c_pct_rank(total_market_cap, ascending := true) AS mcap_pct,
            ({numerator_expr}) / NULLIF(({denominator_expr}), 0) AS exp_wgt_return_6m
        FROM cn_stock_prefactors
        WHERE date >= '{calc_start_date}'
          AND date <= '{END_DATE}'
          AND list_sector IN (1, 2, 3)
          AND is_risk_warning = 0
          AND suspended = 0
          AND list_days >= {MIN_LIST_DAYS}
          AND close IS NOT NULL
          AND turn IS NOT NULL
          AND total_market_cap IS NOT NULL
          AND cs_level1_name IS NOT NULL
          AND instrument NOT LIKE '688%.SH'
          AND instrument NOT LIKE '689%.SH'
          AND instrument NOT LIKE '300%.SZ'
          AND instrument NOT LIKE '301%.SZ'
    ),
    with_mcap_group AS (
        SELECT
            *,
            CASE
                WHEN mcap_pct IS NULL THEN NULL
                WHEN mcap_pct >= 1.0 THEN {MCAP_GROUP_COUNT}
                ELSE CAST(FLOOR(mcap_pct * {MCAP_GROUP_COUNT}) + 1 AS INTEGER)
            END AS mcap_group
        FROM factor_base
    ),
    universe AS (
        SELECT
            date,
            instrument,
            total_market_cap,
            mcap_group,
            exp_wgt_return_6m
        FROM with_mcap_group
        WHERE date IN ({rebalance_date_sql})
          AND mcap_group IN ({selected_group_sql})
          AND cs_level1_name NOT IN ({industry_sql})
          AND exp_wgt_return_6m IS NOT NULL
    ),
    ranked AS (
        SELECT
            date,
            instrument,
            mcap_group,
            exp_wgt_return_6m,
            COUNT(*) OVER (
                PARTITION BY date, mcap_group
            ) AS group_stock_count,
            ROW_NUMBER() OVER (
                PARTITION BY date, mcap_group
                ORDER BY exp_wgt_return_6m ASC, total_market_cap ASC, instrument ASC
            ) AS factor_rank_in_group
        FROM universe
    ),
    selected AS (
        SELECT
            *,
            CAST(CEIL(group_stock_count * {GROUP_SELECT_PCT:.12g}) AS INTEGER)
                AS group_select_num
        FROM ranked
    )
    SELECT
        date AS rebalance_signal_date,
        instrument,
        mcap_group,
        exp_wgt_return_6m,
        factor_rank_in_group
    FROM selected
    WHERE factor_rank_in_group <= group_select_num
    ORDER BY rebalance_signal_date, mcap_group, factor_rank_in_group, instrument
    """

    target_rebalance_df = dai.query(signal_sql).df()
    if target_rebalance_df.empty:
        raise ValueError("调仓候选池为空，请检查日期、过滤条件或因子字段。")

    target_rebalance_df["rebalance_signal_date"] = (
        pd.to_datetime(target_rebalance_df["rebalance_signal_date"])
        .dt.strftime("%Y-%m-%d")
    )
    target_rebalance_df = target_rebalance_df.sort_values(
        ["rebalance_signal_date", "mcap_group", "factor_rank_in_group", "instrument"]
    ).reset_index(drop=True)

    # 防御性校验：若数据库代码格式与预期不一致，不允许被排除板块进入交易信号。
    excluded_board_mask = target_rebalance_df["instrument"].astype(str).str.match(
        r"^(688|689)\d{3}\.SH$|^(300|301)\d{3}\.SZ$"
    )
    if excluded_board_mask.any():
        invalid_codes = target_rebalance_df.loc[
            excluded_board_mask, "instrument"
        ].head(10).tolist()
        raise AssertionError(f"板块排除校验失败，仍发现禁买股票：{invalid_codes}")

    target_rebalance_df["instrument"] = target_rebalance_df["instrument"].astype(
        "category"
    )
    target_rebalance_df["mcap_group"] = target_rebalance_df["mcap_group"].astype(
        np.int16
    )
    target_rebalance_df["factor_rank_in_group"] = target_rebalance_df[
        "factor_rank_in_group"
    ].astype(np.int32)
    target_rebalance_df["exp_wgt_return_6m"] = target_rebalance_df[
        "exp_wgt_return_6m"
    ].astype(np.float32)

    trade_schedule_df = pd.DataFrame({"date": daily_trend_trade["date"].tolist()})
    effective_starts = np.array(
        rebalance_schedule_df["effective_start_date"].tolist(), dtype=object
    )
    active_index = np.searchsorted(
        effective_starts, trade_schedule_df["date"].values, side="right"
    ) - 1
    trade_schedule_df = trade_schedule_df.loc[active_index >= 0].copy()
    trade_schedule_df["rebalance_signal_date"] = np.array(
        rebalance_signal_dates, dtype=object
    )[active_index[active_index >= 0]]

    daily_target_df = trade_schedule_df.merge(
        target_rebalance_df, on="rebalance_signal_date", how="inner"
    ).merge(
        daily_trend_trade[
            ["date", "trend_signal_date", "risk_on", "trend_total_weight"]
        ],
        on="date",
        how="left",
    )
    if daily_target_df.empty:
        raise ValueError("每日目标持仓为空，请检查调仓日与执行日映射。")
    if daily_target_df[["risk_on", "trend_total_weight"]].isna().any().any():
        raise ValueError("每日目标持仓存在缺失的趋势信号。")

    daily_target_df["stock_count"] = daily_target_df.groupby(
        "date", observed=True
    )["instrument"].transform("count")
    daily_target_df["weight"] = (
        daily_target_df["trend_total_weight"] / daily_target_df["stock_count"]
    )

    signal_df = daily_target_df[
        [
            "date",
            "instrument",
            "weight",
            "rebalance_signal_date",
            "trend_signal_date",
            "risk_on",
            "trend_total_weight",
        ]
    ].copy()
    signal_df["date"] = signal_df["date"].astype(str)
    signal_df["instrument"] = signal_df["instrument"].astype(str)
    signal_df["rebalance_signal_date"] = signal_df[
        "rebalance_signal_date"
    ].astype(str)
    signal_df["trend_signal_date"] = signal_df["trend_signal_date"].astype(str)
    signal_df["weight"] = signal_df["weight"].astype(np.float32)
    signal_df["risk_on"] = signal_df["risk_on"].astype(np.int8)
    signal_df["trend_total_weight"] = signal_df["trend_total_weight"].astype(
        np.float32
    )
    signal_df = signal_df.sort_values(["date", "instrument"]).reset_index(drop=True)

    if signal_df.empty:
        raise ValueError("signal_df 为空。")
    all_instruments = sorted(signal_df["instrument"].unique().tolist())
    if not all_instruments:
        raise ValueError("没有可回测标的。")

    effective_end_date = str(signal_df["date"].max())
    return signal_df, all_instruments, effective_end_date


# =====================================================
# 4. BigTrader 交易回调
# =====================================================

def _current_bar_values(data, instrument):
    try:
        row = data.current(instrument, ["open", "upper_limit", "lower_limit", "volume"])
        open_price = _safe_float(row["open"])
        upper_limit = _safe_float(row["upper_limit"])
        lower_limit = _safe_float(row["lower_limit"])
        volume = _safe_float(row["volume"], default=0.0)
    except Exception:
        return None

    if pd.isna(open_price) or pd.isna(upper_limit) or pd.isna(lower_limit):
        return None
    return {
        "open": open_price,
        "upper_limit": upper_limit,
        "lower_limit": lower_limit,
        "volume": volume,
    }


def _can_buy_at_current_open(data, instrument):
    bar = _current_bar_values(data, instrument)
    return bar is not None and bar["volume"] > 0 and bar["open"] < bar["upper_limit"] - EPS


def _can_sell_at_current_open(data, instrument):
    bar = _current_bar_values(data, instrument)
    return bar is not None and bar["volume"] > 0 and bar["open"] > bar["lower_limit"] + EPS


def _build_target_maps(signal_data):
    target_by_date = {}
    meta_by_date = {}
    meta_columns = [
        "rebalance_signal_date",
        "trend_signal_date",
        "risk_on",
        "trend_total_weight",
    ]
    for date, group in signal_data.groupby("date", sort=False):
        target_by_date[date] = dict(
            zip(group["instrument"].values, group["weight"].astype(float).values)
        )
        first_row = group.iloc[0]
        meta_by_date[date] = {column: first_row[column] for column in meta_columns}
    return target_by_date, meta_by_date


def _set_current_bar_matching(context):
    if not USE_CURRENT_BAR_MATCHING:
        return
    try:
        vmatch_enum = getattr(bigtrader, "VMatchAt", None)
        if vmatch_enum is None:
            from bigtrader.constant import VMatchAt

            vmatch_enum = VMatchAt
        if hasattr(vmatch_enum, "CURRENT_BAR"):
            context.set_vmatch_at(vmatch_enum.CURRENT_BAR)
        elif hasattr(vmatch_enum, "CURRENT"):
            context.set_vmatch_at(vmatch_enum.CURRENT)
        else:
            raise AttributeError("未找到当前 Bar 撮合枚举。")
    except Exception as error:
        print("警告：未能设置当前 Bar 撮合模式：", repr(error))


def _load_desired_weight_map(context):
    try:
        stored = context.user_store.get("desired_weight_map", {})
        return dict(stored) if isinstance(stored, dict) else {}
    except Exception:
        return {}


def _save_desired_weight_map(context):
    try:
        context.user_store["desired_weight_map"] = context.desired_weight_map
    except Exception:
        pass


def _order_succeeded(result):
    return result is None or result == 0


def initialize(context):
    signal_data = context.data.copy()
    signal_data["date"] = signal_data["date"].astype(str)
    signal_data["instrument"] = signal_data["instrument"].astype(str)
    context.target_by_date, context.meta_by_date = _build_target_maps(signal_data)
    context.desired_weight_map = _load_desired_weight_map(context)
    context.set_commission(
        bigtrader.PerOrder(
            buy_cost=BUY_COST,
            sell_cost=SELL_COST,
            min_cost=MIN_COST,
        )
    )
    _set_current_bar_matching(context)


def handle_data(context, data):
    today = data.current_dt.strftime("%Y-%m-%d")
    target_map = context.target_by_date.get(today)
    if target_map is None:
        return

    if VERBOSE:
        meta = context.meta_by_date.get(today, {})
        print(
            f"{today} 执行：调仓信号={meta.get('rebalance_signal_date')}，"
            f"趋势信号={meta.get('trend_signal_date')}，"
            f"风险开关={meta.get('risk_on')}，"
            f"目标总仓位={float(meta.get('trend_total_weight', 0.0)):.2%}，"
            f"目标股票数={len(target_map)}"
        )

    holding_instruments = set(context.get_positions().keys())
    target_instruments = set(target_map.keys())

    # 不在目标池内的持仓，若交易条件允许则卖出。
    for instrument in sorted(holding_instruments - target_instruments):
        if not _can_sell_at_current_open(data, instrument):
            continue
        result = context.order_target_percent(instrument, 0)
        if _order_succeeded(result):
            context.desired_weight_map[instrument] = 0.0

    # 目标股票按目标权重调整；被涨跌停或停牌阻断时保留原仓位/现金，不再分配给其他股票。
    for instrument, target_weight in target_map.items():
        target_weight = float(target_weight)
        previous_weight = float(context.desired_weight_map.get(instrument, 0.0))
        weight_change = target_weight - previous_weight
        if abs(weight_change) < 1e-8:
            continue

        tradable = (
            _can_buy_at_current_open(data, instrument)
            if weight_change > 0
            else _can_sell_at_current_open(data, instrument)
        )
        if not tradable:
            continue

        result = context.order_target_percent(instrument, target_weight)
        if _order_succeeded(result):
            context.desired_weight_map[instrument] = target_weight

    _save_desired_weight_map(context)


# =====================================================
# 5. 运行回测
# =====================================================

signal_df, all_target_instruments, EFFECTIVE_END_DATE = build_signal_data()

print(f"查询结束日 END_DATE：{END_DATE}")
print(f"实际运行结束日 EFFECTIVE_END_DATE：{EFFECTIVE_END_DATE}")
print(f"初始资金：{CAPITAL_BASE:,.0f} 元")
print("已排除：科创板 688/689，创业板 300/301")
print(
    f"传入 BigTrader 的交易日数量：{signal_df['date'].nunique()}，"
    f"股票数量：{len(all_target_instruments)}"
)

performance = bigtrader.run(
    market=bigtrader.Market.CN_STOCK,
    frequency=bigtrader.Frequency.DAILY,
    start_date=START_DATE,
    end_date=EFFECTIVE_END_DATE,
    capital_base=CAPITAL_BASE,
    instruments=all_target_instruments,
    data=signal_df,
    initialize=initialize,
    handle_data=handle_data,
    benchmark=BENCHMARK,
    order_price_field_buy="open",
    order_price_field_sell="open",
    volume_limit=0.025,
)


查询结束日 END_DATE：2026-09-14
实际运行结束日 EFFECTIVE_END_DATE：2026-09-14
初始资金：100,000 元
已排除：科创板 688/689，创业板 300/301
传入 BigTrader 的交易日数量：896，股票数量：141
[2026-09-14 23:09:30] [info     ] bigtrader init ..
[2026-09-14 23:09:30] [info     ] bigtrader.run start: market=cn_stock, frequency=1d, mode=backtest, account_type=STOCK, start_date=2023-01-01, end_date=2026-09-14
[2026-09-14 23:09:31] [info     ] bigtrader<backtest> init ..
[2026-09-14 23:09:31] [info     ] prepare data ..
[2026-09-14 23:09:31] [info     ] bar1d_df: (129988, 16)
[2026-09-14 23:09:31] [info     ] bigtrader use dividend data: (200, 8)
[2026-09-14 23:09:31] [info     ] bigtrader run ..


[2026-09-14 23:09:33] [info     ] bigtrader run done.


In [3]:
# =====================================================
# exp_wgt_return_6m 日频模拟策略：15 市值组 + 指定组别内因子低分位等权
#
# 与原模拟策略保持一致的部分：
# - 市值分组、因子计算、行业过滤、趋势仓位与 20 日调仓规则
# - T 日收盘后形成信号，T+1 日开盘执行
# - 开盘涨跌停、停牌与成交量约束
#
# 本模拟版相对原模拟策略的调整：
# 1) 初始资金改为 100,000 元；
# 2) 候选池仅允许普通沪深 A 股主板，从而排除所有需要额外交易权限
#    或资金/交易经验门槛的板块，包括科创板、创业板、北交所和新三板。
# =====================================================

import math
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import dai
from bigquant import bigtrader


# =====================================================
# 1. 策略参数区
# =====================================================

# 回测起点同时决定固定的 20 日调仓相位；提交模拟后不要频繁改动。
START_DATE = "2023-01-01"

# 模拟策略的查询终点自动取运行当天；平台每日运行会自然向后扩展。
END_DATE = datetime.today().strftime("%Y-%m-%d")

MCAP_GROUP_COUNT = 15
SELECT_MCAP_GROUPS = [1]
GROUP_SELECT_PCT = 0.10
REBALANCE_DAYS = 20

TARGET_TOTAL_WEIGHT = 0.98
DEFENSIVE_TOTAL_WEIGHT = 0.30
TREND_INDEX = "000852.SH"
TREND_MA_DAYS = 30

FACTOR_MONTHS = 6
LOOKBACK_DAYS = 21 * FACTOR_MONTHS
BEFORE_START_DAYS = max(300, TREND_MA_DAYS * 3)
MIN_LIST_DAYS = int(LOOKBACK_DAYS * 365 / 252) + 30

# 模拟策略初始资金：10 万元。
CAPITAL_BASE = 100_000

EXCLUDE_INDUSTRIES = [
    "交通运输",
    "电力及公用事业",
    "纺织服装",
    "轻工制造",
    "家电",
    "石油石化",
    "综合",
    "银行",
]

BENCHMARK = "000300.SH"
VERBOSE = False
BUY_COST = 0.0003
SELL_COST = 0.0013
# 单笔最低佣金：5 元。
MIN_COST = 5.0
USE_CURRENT_BAR_MATCHING = True
EPS = 1e-10


# =====================================================
# 2. 参数与通用辅助函数
# =====================================================

def _normalize_mcap_groups(groups, group_count):
    if groups is None or len(groups) == 0:
        raise ValueError("SELECT_MCAP_GROUPS 不能为空，例如 [1, 2, 3]。")

    normalized = sorted({int(g) for g in groups})
    invalid = [g for g in normalized if g < 1 or g > group_count]
    if invalid:
        raise ValueError(
            f"SELECT_MCAP_GROUPS 中存在非法组别 {invalid}；"
            f"有效范围为 1 到 {group_count}。"
        )
    return normalized


def _sql_quote(values):
    if values is None or len(values) == 0:
        raise ValueError("SQL 列表不能为空。")
    return ", ".join("'" + str(x).replace("'", "''") + "'" for x in values)


def _sql_int_list(values):
    if values is None or len(values) == 0:
        raise ValueError("SQL 整数列表不能为空。")
    return ", ".join(str(int(x)) for x in values)


def _safe_float(value, default=np.nan):
    try:
        if value is None or pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def _validate_parameters():
    global SELECT_MCAP_GROUPS
    SELECT_MCAP_GROUPS = _normalize_mcap_groups(
        SELECT_MCAP_GROUPS, MCAP_GROUP_COUNT
    )

    if not 0 < GROUP_SELECT_PCT <= 1:
        raise ValueError("GROUP_SELECT_PCT 必须位于 (0, 1] 区间内。")
    if REBALANCE_DAYS <= 0:
        raise ValueError("REBALANCE_DAYS 必须为正整数。")
    if not 0 <= DEFENSIVE_TOTAL_WEIGHT <= TARGET_TOTAL_WEIGHT <= 1:
        raise ValueError("仓位参数必须满足 0 <= DEFENSIVE <= TARGET <= 1。")


def _lag(field, periods):
    return field if periods == 0 else f"m_lag({field}, {periods})"


def _build_exp_wgt_return_expr():
    """构造与原策略相同的 6 个月成交量加权衰减收益表达式。"""
    numerator_terms = []
    denominator_terms = []

    for i in range(LOOKBACK_DAYS):
        decay_weight = math.exp(-i / FACTOR_MONTHS / 4.0)
        close_i = _lag("close", i)
        close_i_1 = _lag("close", i + 1)
        turn_i = _lag("turn", i)
        daily_return = f"(({close_i} / NULLIF({close_i_1}, 0)) - 1.0)"

        numerator_terms.append(
            f"COALESCE(({decay_weight:.12g} * {turn_i} * {daily_return}), 0.0)"
        )
        denominator_terms.append(
            f"COALESCE(({decay_weight:.12g} * {turn_i}), 0.0)"
        )

    return " + ".join(numerator_terms), " + ".join(denominator_terms)


# =====================================================
# 3. 信号构造
# =====================================================

def build_signal_data():
    _validate_parameters()

    calc_start_date = (
        datetime.strptime(START_DATE, "%Y-%m-%d")
        - timedelta(days=BEFORE_START_DAYS)
    ).strftime("%Y-%m-%d")

    industry_sql = _sql_quote(EXCLUDE_INDUSTRIES)
    selected_group_sql = _sql_int_list(SELECT_MCAP_GROUPS)
    numerator_expr, denominator_expr = _build_exp_wgt_return_expr()

    # 每天的趋势结果都只使用当日收盘前已有数据；在下一交易日才会生效。
    trend_sql = f"""
    WITH index_raw AS (
        SELECT
            date,
            close AS trend_index_close,
            AVG(close) OVER (
                PARTITION BY instrument
                ORDER BY date
                ROWS BETWEEN {TREND_MA_DAYS - 1} PRECEDING AND CURRENT ROW
            ) AS trend_index_ma
        FROM cn_stock_index_bar1d
        WHERE instrument = '{TREND_INDEX}'
          AND date >= '{calc_start_date}'
          AND date <= '{END_DATE}'
    )
    SELECT
        date,
        trend_index_close,
        trend_index_ma,
        CASE
            WHEN trend_index_ma IS NULL THEN 0
            WHEN trend_index_close > trend_index_ma THEN 1
            ELSE 0
        END AS risk_on,
        CASE
            WHEN trend_index_ma IS NULL THEN {DEFENSIVE_TOTAL_WEIGHT}
            WHEN trend_index_close > trend_index_ma THEN {TARGET_TOTAL_WEIGHT}
            ELSE {DEFENSIVE_TOTAL_WEIGHT}
        END AS trend_total_weight
    FROM index_raw
    WHERE date >= '{START_DATE}'
    ORDER BY date ASC
    """

    daily_trend_raw = dai.query(trend_sql).df()
    if daily_trend_raw.empty:
        raise ValueError("趋势指数数据为空，请检查 TREND_INDEX 与日期区间。")

    daily_trend_raw["date"] = (
        pd.to_datetime(daily_trend_raw["date"]).dt.strftime("%Y-%m-%d")
    )
    daily_trend_raw = daily_trend_raw.sort_values("date").reset_index(drop=True)
    all_dates = daily_trend_raw["date"].tolist()
    if len(all_dates) < 2:
        raise ValueError("交易日数量不足，无法形成 T 日信号、T+1 日执行。")

    # 交易日 t 只能使用 t-1 的收盘趋势信号。
    daily_trend_trade = daily_trend_raw.copy()
    daily_trend_trade["trend_signal_date"] = daily_trend_trade["date"].shift(1)
    for column in [
        "trend_index_close",
        "trend_index_ma",
        "risk_on",
        "trend_total_weight",
    ]:
        daily_trend_trade[column] = daily_trend_trade[column].shift(1)

    daily_trend_trade = daily_trend_trade.dropna(
        subset=["trend_signal_date", "trend_total_weight"]
    ).copy()
    daily_trend_trade["risk_on"] = daily_trend_trade["risk_on"].astype(np.int8)
    daily_trend_trade["trend_total_weight"] = daily_trend_trade[
        "trend_total_weight"
    ].astype(np.float32)

    # 调仓信号由固定相位产生，且在下一交易日执行。
    rebalance_signal_dates = all_dates[::REBALANCE_DAYS]
    next_date_map = {
        all_dates[i]: all_dates[i + 1] for i in range(len(all_dates) - 1)
    }
    rebalance_signal_dates = [
        date for date in rebalance_signal_dates if date in next_date_map
    ]
    if not rebalance_signal_dates:
        raise ValueError("调仓日期为空，请检查 START_DATE 或 REBALANCE_DAYS。")

    rebalance_date_sql = _sql_quote(rebalance_signal_dates)
    rebalance_schedule_df = pd.DataFrame(
        {
            "rebalance_signal_date": rebalance_signal_dates,
            "effective_start_date": [
                next_date_map[date] for date in rebalance_signal_dates
            ],
        }
    )

    # 白名单规则放在 factor_base 中：仅让普通沪深 A 股主板参与市值分位。
    # 因此科创板、创业板、北交所、新三板和 B 股均不会参与后续排序或交易。
    signal_sql = f"""
    WITH factor_base AS (
        SELECT
            date,
            instrument,
            cs_level1_name,
            total_market_cap,
            c_pct_rank(total_market_cap, ascending := true) AS mcap_pct,
            ({numerator_expr}) / NULLIF(({denominator_expr}), 0) AS exp_wgt_return_6m
        FROM cn_stock_prefactors
        WHERE date >= '{calc_start_date}'
          AND date <= '{END_DATE}'
          AND list_sector IN (1, 2, 3)
          AND is_risk_warning = 0
          AND suspended = 0
          AND list_days >= {MIN_LIST_DAYS}
          AND close IS NOT NULL
          AND turn IS NOT NULL
          AND total_market_cap IS NOT NULL
          AND cs_level1_name IS NOT NULL
          AND (
              instrument LIKE '600%.SH'
              OR instrument LIKE '601%.SH'
              OR instrument LIKE '603%.SH'
              OR instrument LIKE '605%.SH'
              OR instrument LIKE '000%.SZ'
              OR instrument LIKE '001%.SZ'
              OR instrument LIKE '002%.SZ'
              OR instrument LIKE '003%.SZ'
          )
    ),
    with_mcap_group AS (
        SELECT
            *,
            CASE
                WHEN mcap_pct IS NULL THEN NULL
                WHEN mcap_pct >= 1.0 THEN {MCAP_GROUP_COUNT}
                ELSE CAST(FLOOR(mcap_pct * {MCAP_GROUP_COUNT}) + 1 AS INTEGER)
            END AS mcap_group
        FROM factor_base
    ),
    universe AS (
        SELECT
            date,
            instrument,
            total_market_cap,
            mcap_group,
            exp_wgt_return_6m
        FROM with_mcap_group
        WHERE date IN ({rebalance_date_sql})
          AND mcap_group IN ({selected_group_sql})
          AND cs_level1_name NOT IN ({industry_sql})
          AND exp_wgt_return_6m IS NOT NULL
    ),
    ranked AS (
        SELECT
            date,
            instrument,
            mcap_group,
            exp_wgt_return_6m,
            COUNT(*) OVER (
                PARTITION BY date, mcap_group
            ) AS group_stock_count,
            ROW_NUMBER() OVER (
                PARTITION BY date, mcap_group
                ORDER BY exp_wgt_return_6m ASC, total_market_cap ASC, instrument ASC
            ) AS factor_rank_in_group
        FROM universe
    ),
    selected AS (
        SELECT
            *,
            CAST(CEIL(group_stock_count * {GROUP_SELECT_PCT:.12g}) AS INTEGER)
                AS group_select_num
        FROM ranked
    )
    SELECT
        date AS rebalance_signal_date,
        instrument,
        mcap_group,
        exp_wgt_return_6m,
        factor_rank_in_group
    FROM selected
    WHERE factor_rank_in_group <= group_select_num
    ORDER BY rebalance_signal_date, mcap_group, factor_rank_in_group, instrument
    """

    target_rebalance_df = dai.query(signal_sql).df()
    if target_rebalance_df.empty:
        raise ValueError("调仓候选池为空，请检查日期、过滤条件或因子字段。")

    target_rebalance_df["rebalance_signal_date"] = (
        pd.to_datetime(target_rebalance_df["rebalance_signal_date"])
        .dt.strftime("%Y-%m-%d")
    )
    target_rebalance_df = target_rebalance_df.sort_values(
        ["rebalance_signal_date", "mcap_group", "factor_rank_in_group", "instrument"]
    ).reset_index(drop=True)

    # 二次校验：任何不属于普通沪深 A 股主板的代码均不得进入交易信号。
    ordinary_a_share_mask = target_rebalance_df["instrument"].astype(str).str.match(
        r"^(600|601|603|605)\d{3}\.SH$|^(000|001|002|003)\d{3}\.SZ$"
    )
    if not ordinary_a_share_mask.all():
        invalid_codes = target_rebalance_df.loc[
            ~ordinary_a_share_mask, "instrument"
        ].head(10).tolist()
        raise AssertionError(f"普通沪深 A 股主板白名单校验失败：{invalid_codes}")

    target_rebalance_df["instrument"] = target_rebalance_df["instrument"].astype(
        "category"
    )
    target_rebalance_df["mcap_group"] = target_rebalance_df["mcap_group"].astype(
        np.int16
    )
    target_rebalance_df["factor_rank_in_group"] = target_rebalance_df[
        "factor_rank_in_group"
    ].astype(np.int32)
    target_rebalance_df["exp_wgt_return_6m"] = target_rebalance_df[
        "exp_wgt_return_6m"
    ].astype(np.float32)

    trade_schedule_df = pd.DataFrame({"date": daily_trend_trade["date"].tolist()})
    effective_starts = np.array(
        rebalance_schedule_df["effective_start_date"].tolist(), dtype=object
    )
    active_index = np.searchsorted(
        effective_starts, trade_schedule_df["date"].values, side="right"
    ) - 1
    trade_schedule_df = trade_schedule_df.loc[active_index >= 0].copy()
    trade_schedule_df["rebalance_signal_date"] = np.array(
        rebalance_signal_dates, dtype=object
    )[active_index[active_index >= 0]]

    daily_target_df = trade_schedule_df.merge(
        target_rebalance_df, on="rebalance_signal_date", how="inner"
    ).merge(
        daily_trend_trade[
            ["date", "trend_signal_date", "risk_on", "trend_total_weight"]
        ],
        on="date",
        how="left",
    )
    if daily_target_df.empty:
        raise ValueError("每日目标持仓为空，请检查调仓日与执行日映射。")
    if daily_target_df[["risk_on", "trend_total_weight"]].isna().any().any():
        raise ValueError("每日目标持仓存在缺失的趋势信号。")

    daily_target_df["stock_count"] = daily_target_df.groupby(
        "date", observed=True
    )["instrument"].transform("count")
    daily_target_df["weight"] = (
        daily_target_df["trend_total_weight"] / daily_target_df["stock_count"]
    )

    signal_df = daily_target_df[
        [
            "date",
            "instrument",
            "weight",
            "rebalance_signal_date",
            "trend_signal_date",
            "risk_on",
            "trend_total_weight",
        ]
    ].copy()
    signal_df["date"] = signal_df["date"].astype(str)
    signal_df["instrument"] = signal_df["instrument"].astype(str)
    signal_df["rebalance_signal_date"] = signal_df[
        "rebalance_signal_date"
    ].astype(str)
    signal_df["trend_signal_date"] = signal_df["trend_signal_date"].astype(str)
    signal_df["weight"] = signal_df["weight"].astype(np.float32)
    signal_df["risk_on"] = signal_df["risk_on"].astype(np.int8)
    signal_df["trend_total_weight"] = signal_df["trend_total_weight"].astype(
        np.float32
    )
    signal_df = signal_df.sort_values(["date", "instrument"]).reset_index(drop=True)

    if signal_df.empty:
        raise ValueError("signal_df 为空。")
    all_instruments = sorted(signal_df["instrument"].unique().tolist())
    if not all_instruments:
        raise ValueError("没有可回测标的。")

    effective_end_date = str(signal_df["date"].max())
    return signal_df, all_instruments, effective_end_date


# =====================================================
# 4. BigTrader 交易回调
# =====================================================

def _current_bar_values(data, instrument):
    try:
        row = data.current(instrument, ["open", "upper_limit", "lower_limit", "volume"])
        open_price = _safe_float(row["open"])
        upper_limit = _safe_float(row["upper_limit"])
        lower_limit = _safe_float(row["lower_limit"])
        volume = _safe_float(row["volume"], default=0.0)
    except Exception:
        return None

    if pd.isna(open_price) or pd.isna(upper_limit) or pd.isna(lower_limit):
        return None
    return {
        "open": open_price,
        "upper_limit": upper_limit,
        "lower_limit": lower_limit,
        "volume": volume,
    }


def _can_buy_at_current_open(data, instrument):
    bar = _current_bar_values(data, instrument)
    return bar is not None and bar["volume"] > 0 and bar["open"] < bar["upper_limit"] - EPS


def _can_sell_at_current_open(data, instrument):
    bar = _current_bar_values(data, instrument)
    return bar is not None and bar["volume"] > 0 and bar["open"] > bar["lower_limit"] + EPS


def _build_target_maps(signal_data):
    target_by_date = {}
    meta_by_date = {}
    meta_columns = [
        "rebalance_signal_date",
        "trend_signal_date",
        "risk_on",
        "trend_total_weight",
    ]
    for date, group in signal_data.groupby("date", sort=False):
        target_by_date[date] = dict(
            zip(group["instrument"].values, group["weight"].astype(float).values)
        )
        first_row = group.iloc[0]
        meta_by_date[date] = {column: first_row[column] for column in meta_columns}
    return target_by_date, meta_by_date


def _set_current_bar_matching(context):
    if not USE_CURRENT_BAR_MATCHING:
        return
    try:
        vmatch_enum = getattr(bigtrader, "VMatchAt", None)
        if vmatch_enum is None:
            from bigtrader.constant import VMatchAt

            vmatch_enum = VMatchAt
        if hasattr(vmatch_enum, "CURRENT_BAR"):
            context.set_vmatch_at(vmatch_enum.CURRENT_BAR)
        elif hasattr(vmatch_enum, "CURRENT"):
            context.set_vmatch_at(vmatch_enum.CURRENT)
        else:
            raise AttributeError("未找到当前 Bar 撮合枚举。")
    except Exception as error:
        print("警告：未能设置当前 Bar 撮合模式：", repr(error))


def _load_desired_weight_map(context):
    try:
        stored = context.user_store.get("desired_weight_map", {})
        return dict(stored) if isinstance(stored, dict) else {}
    except Exception:
        return {}


def _save_desired_weight_map(context):
    try:
        context.user_store["desired_weight_map"] = context.desired_weight_map
    except Exception:
        pass


def _order_succeeded(result):
    return result is None or result == 0


def initialize(context):
    signal_data = context.data.copy()
    signal_data["date"] = signal_data["date"].astype(str)
    signal_data["instrument"] = signal_data["instrument"].astype(str)
    context.target_by_date, context.meta_by_date = _build_target_maps(signal_data)
    context.desired_weight_map = _load_desired_weight_map(context)
    context.set_commission(
        bigtrader.PerOrder(
            buy_cost=BUY_COST,
            sell_cost=SELL_COST,
            min_cost=MIN_COST,
        )
    )
    _set_current_bar_matching(context)


def handle_data(context, data):
    today = data.current_dt.strftime("%Y-%m-%d")
    target_map = context.target_by_date.get(today)
    if target_map is None:
        return

    if VERBOSE:
        meta = context.meta_by_date.get(today, {})
        print(
            f"{today} 执行：调仓信号={meta.get('rebalance_signal_date')}，"
            f"趋势信号={meta.get('trend_signal_date')}，"
            f"风险开关={meta.get('risk_on')}，"
            f"目标总仓位={float(meta.get('trend_total_weight', 0.0)):.2%}，"
            f"目标股票数={len(target_map)}"
        )

    holding_instruments = set(context.get_positions().keys())
    target_instruments = set(target_map.keys())

    # 不在目标池内的持仓，若交易条件允许则卖出。
    for instrument in sorted(holding_instruments - target_instruments):
        if not _can_sell_at_current_open(data, instrument):
            continue
        result = context.order_target_percent(instrument, 0)
        if _order_succeeded(result):
            context.desired_weight_map[instrument] = 0.0

    # 目标股票按目标权重调整；被涨跌停或停牌阻断时保留原仓位/现金，不再分配给其他股票。
    for instrument, target_weight in target_map.items():
        target_weight = float(target_weight)
        previous_weight = float(context.desired_weight_map.get(instrument, 0.0))
        weight_change = target_weight - previous_weight
        if abs(weight_change) < 1e-8:
            continue

        tradable = (
            _can_buy_at_current_open(data, instrument)
            if weight_change > 0
            else _can_sell_at_current_open(data, instrument)
        )
        if not tradable:
            continue

        result = context.order_target_percent(instrument, target_weight)
        if _order_succeeded(result):
            context.desired_weight_map[instrument] = target_weight

    _save_desired_weight_map(context)


# =====================================================
# 5. 运行回测并提交日频模拟
# =====================================================

signal_df, all_target_instruments, EFFECTIVE_END_DATE = build_signal_data()

print(f"查询结束日 END_DATE：{END_DATE}")
print(f"实际运行结束日 EFFECTIVE_END_DATE：{EFFECTIVE_END_DATE}")
print(f"模拟初始资金：{CAPITAL_BASE:,.0f} 元")
print("交易白名单：仅普通沪深 A 股主板；已排除科创板、创业板、北交所、新三板及 B 股")
print(
    f"传入 BigTrader 的交易日数量：{signal_df['date'].nunique()}，"
    f"股票数量：{len(all_target_instruments)}"
)

performance = bigtrader.run(
    market=bigtrader.Market.CN_STOCK,
    frequency=bigtrader.Frequency.DAILY,
    start_date=START_DATE,
    end_date=EFFECTIVE_END_DATE,
    capital_base=CAPITAL_BASE,
    instruments=all_target_instruments,
    data=signal_df,
    initialize=initialize,
    handle_data=handle_data,
    benchmark=BENCHMARK,
    order_price_field_buy="open",
    order_price_field_sell="open",
    volume_limit=0.025,
)


查询结束日 END_DATE：2026-09-14
实际运行结束日 EFFECTIVE_END_DATE：2026-09-14
模拟初始资金：100,000 元
交易白名单：仅普通沪深 A 股主板；已排除科创板、创业板、北交所、新三板及 B 股
传入 BigTrader 的交易日数量：896，股票数量：141
[2026-09-14 23:27:21] [info     ] bigtrader init ..
[2026-09-14 23:27:21] [info     ] bigtrader.run start: market=cn_stock, frequency=1d, mode=backtest, account_type=STOCK, start_date=2023-01-01, end_date=2026-09-14
[2026-09-14 23:27:22] [info     ] bigtrader<backtest> init ..
[2026-09-14 23:27:22] [info     ] prepare data ..
[2026-09-14 23:27:22] [info     ] bar1d_df: (129988, 16)
[2026-09-14 23:27:22] [info     ] bigtrader use dividend data: (200, 8)
[2026-09-14 23:27:22] [info     ] bigtrader run ..


[2026-09-14 23:27:23] [info     ] bigtrader run done.


In [2]:
# =====================================================
# exp_wgt_return_6m 回测：15 市值组 + 指定组别内因子低分位等权
#
# 与原模拟策略保持一致的部分：
# - 市值分组、因子计算、行业过滤、趋势仓位与 20 日调仓规则
# - T 日收盘后形成信号，T+1 日开盘执行
# - 开盘涨跌停、停牌与成交量约束
#
# 本回测版仅作两项调整：
# 1) 初始资金改为 100,000 元；
# 2) 候选池在市值分组前剔除科创板（688/689）与创业板（300/301）。
# =====================================================

import math
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import dai
from bigquant import bigtrader


# =====================================================
# 1. 策略参数区
# =====================================================

# 回测起点同时决定固定的 20 日调仓相位；如需和模拟策略对照，请不要改变。
START_DATE = "2023-01-01"

# 回测终点。默认取运行当天；若需要固定样本期，可改为如 "2026-09-11"。
END_DATE = datetime.today().strftime("%Y-%m-%d")

MCAP_GROUP_COUNT = 15
SELECT_MCAP_GROUPS = [1]
GROUP_SELECT_PCT = 0.10
REBALANCE_DAYS = 20

TARGET_TOTAL_WEIGHT = 0.98
# 防御状态下，原收益信号仅保留 10% 仓位。
DEFENSIVE_TOTAL_WEIGHT = 0.10
TREND_INDEX = "000852.SH"
# T 日收盘跌破 MA60 后，T+1 开盘进入防御；重新站上 MA60 后，T+1 开盘恢复原信号仓位。
TREND_MA_DAYS = 30

# 防御状态的收益补偿资产：工商银行、交通银行、中国银行。
# 它们不参与收益因子候选池；只在防御状态下等权配置。
DEFENSIVE_COMPENSATION_INSTRUMENTS = [
    "601398.SH",  # 工商银行
    "601328.SH",  # 交通银行
    "601988.SH",  # 中国银行
]

FACTOR_MONTHS = 6
LOOKBACK_DAYS = 21 * FACTOR_MONTHS
BEFORE_START_DAYS = max(300, TREND_MA_DAYS * 3)
MIN_LIST_DAYS = int(LOOKBACK_DAYS * 365 / 252) + 30

# 本次回测初始资金：10 万元。
CAPITAL_BASE = 100_000

EXCLUDE_INDUSTRIES = [
    "交通运输",
    "电力及公用事业",
    "纺织服装",
    "轻工制造",
    "家电",
    "石油石化",
    "综合",
    "银行",
]

BENCHMARK = "000300.SH"
VERBOSE = False
BUY_COST = 0.0003
SELL_COST = 0.0013
# 不设单笔最低佣金：按买入/卖出费率实际计费。
MIN_COST = 0.0
USE_CURRENT_BAR_MATCHING = True
EPS = 1e-10


# =====================================================
# 2. 参数与通用辅助函数
# =====================================================

def _normalize_mcap_groups(groups, group_count):
    if groups is None or len(groups) == 0:
        raise ValueError("SELECT_MCAP_GROUPS 不能为空，例如 [1, 2, 3]。")

    normalized = sorted({int(g) for g in groups})
    invalid = [g for g in normalized if g < 1 or g > group_count]
    if invalid:
        raise ValueError(
            f"SELECT_MCAP_GROUPS 中存在非法组别 {invalid}；"
            f"有效范围为 1 到 {group_count}。"
        )
    return normalized


def _sql_quote(values):
    if values is None or len(values) == 0:
        raise ValueError("SQL 列表不能为空。")
    return ", ".join("'" + str(x).replace("'", "''") + "'" for x in values)


def _sql_int_list(values):
    if values is None or len(values) == 0:
        raise ValueError("SQL 整数列表不能为空。")
    return ", ".join(str(int(x)) for x in values)


def _safe_float(value, default=np.nan):
    try:
        if value is None or pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def _validate_parameters():
    global SELECT_MCAP_GROUPS
    SELECT_MCAP_GROUPS = _normalize_mcap_groups(
        SELECT_MCAP_GROUPS, MCAP_GROUP_COUNT
    )

    if not 0 < GROUP_SELECT_PCT <= 1:
        raise ValueError("GROUP_SELECT_PCT 必须位于 (0, 1] 区间内。")
    if REBALANCE_DAYS <= 0:
        raise ValueError("REBALANCE_DAYS 必须为正整数。")
    if not 0 <= DEFENSIVE_TOTAL_WEIGHT <= TARGET_TOTAL_WEIGHT <= 1:
        raise ValueError("仓位参数必须满足 0 <= DEFENSIVE <= TARGET <= 1。")
    if not DEFENSIVE_COMPENSATION_INSTRUMENTS:
        raise ValueError("DEFENSIVE_COMPENSATION_INSTRUMENTS 不能为空。")
    if len(set(DEFENSIVE_COMPENSATION_INSTRUMENTS)) != len(
        DEFENSIVE_COMPENSATION_INSTRUMENTS
    ):
        raise ValueError("DEFENSIVE_COMPENSATION_INSTRUMENTS 不能包含重复股票。")


def _lag(field, periods):
    return field if periods == 0 else f"m_lag({field}, {periods})"


def _build_exp_wgt_return_expr():
    """构造与原策略相同的 6 个月成交量加权衰减收益表达式。"""
    numerator_terms = []
    denominator_terms = []

    for i in range(LOOKBACK_DAYS):
        decay_weight = math.exp(-i / FACTOR_MONTHS / 4.0)
        close_i = _lag("close", i)
        close_i_1 = _lag("close", i + 1)
        turn_i = _lag("turn", i)
        daily_return = f"(({close_i} / NULLIF({close_i_1}, 0)) - 1.0)"

        numerator_terms.append(
            f"COALESCE(({decay_weight:.12g} * {turn_i} * {daily_return}), 0.0)"
        )
        denominator_terms.append(
            f"COALESCE(({decay_weight:.12g} * {turn_i}), 0.0)"
        )

    return " + ".join(numerator_terms), " + ".join(denominator_terms)


# =====================================================
# 3. 信号构造
# =====================================================

def build_signal_data():
    _validate_parameters()

    calc_start_date = (
        datetime.strptime(START_DATE, "%Y-%m-%d")
        - timedelta(days=BEFORE_START_DAYS)
    ).strftime("%Y-%m-%d")

    industry_sql = _sql_quote(EXCLUDE_INDUSTRIES)
    selected_group_sql = _sql_int_list(SELECT_MCAP_GROUPS)
    numerator_expr, denominator_expr = _build_exp_wgt_return_expr()

    # 每天的趋势结果都只使用当日收盘前已有数据；在下一交易日才会生效。
    trend_sql = f"""
    WITH index_raw AS (
        SELECT
            date,
            close AS trend_index_close,
            AVG(close) OVER (
                PARTITION BY instrument
                ORDER BY date
                ROWS BETWEEN {TREND_MA_DAYS - 1} PRECEDING AND CURRENT ROW
            ) AS trend_index_ma
        FROM cn_stock_index_bar1d
        WHERE instrument = '{TREND_INDEX}'
          AND date >= '{calc_start_date}'
          AND date <= '{END_DATE}'
    )
    SELECT
        date,
        trend_index_close,
        trend_index_ma,
        CASE
            WHEN trend_index_ma IS NULL THEN 0
            WHEN trend_index_close > trend_index_ma THEN 1
            ELSE 0
        END AS risk_on,
        CASE
            WHEN trend_index_ma IS NULL THEN {DEFENSIVE_TOTAL_WEIGHT}
            WHEN trend_index_close > trend_index_ma THEN {TARGET_TOTAL_WEIGHT}
            ELSE {DEFENSIVE_TOTAL_WEIGHT}
        END AS trend_total_weight
    FROM index_raw
    WHERE date >= '{START_DATE}'
    ORDER BY date ASC
    """

    daily_trend_raw = dai.query(trend_sql).df()
    if daily_trend_raw.empty:
        raise ValueError("趋势指数数据为空，请检查 TREND_INDEX 与日期区间。")

    daily_trend_raw["date"] = (
        pd.to_datetime(daily_trend_raw["date"]).dt.strftime("%Y-%m-%d")
    )
    daily_trend_raw = daily_trend_raw.sort_values("date").reset_index(drop=True)
    all_dates = daily_trend_raw["date"].tolist()
    if len(all_dates) < 2:
        raise ValueError("交易日数量不足，无法形成 T 日信号、T+1 日执行。")

    # 交易日 t 只能使用 t-1 的收盘趋势信号。
    daily_trend_trade = daily_trend_raw.copy()
    daily_trend_trade["trend_signal_date"] = daily_trend_trade["date"].shift(1)
    for column in [
        "trend_index_close",
        "trend_index_ma",
        "risk_on",
        "trend_total_weight",
    ]:
        daily_trend_trade[column] = daily_trend_trade[column].shift(1)

    daily_trend_trade = daily_trend_trade.dropna(
        subset=["trend_signal_date", "trend_total_weight"]
    ).copy()
    daily_trend_trade["risk_on"] = daily_trend_trade["risk_on"].astype(np.int8)
    daily_trend_trade["trend_total_weight"] = daily_trend_trade[
        "trend_total_weight"
    ].astype(np.float32)

    # 调仓信号由固定相位产生，且在下一交易日执行。
    rebalance_signal_dates = all_dates[::REBALANCE_DAYS]
    next_date_map = {
        all_dates[i]: all_dates[i + 1] for i in range(len(all_dates) - 1)
    }
    rebalance_signal_dates = [
        date for date in rebalance_signal_dates if date in next_date_map
    ]
    if not rebalance_signal_dates:
        raise ValueError("调仓日期为空，请检查 START_DATE 或 REBALANCE_DAYS。")

    rebalance_date_sql = _sql_quote(rebalance_signal_dates)
    rebalance_schedule_df = pd.DataFrame(
        {
            "rebalance_signal_date": rebalance_signal_dates,
            "effective_start_date": [
                next_date_map[date] for date in rebalance_signal_dates
            ],
        }
    )

    # 白名单规则放在 factor_base 中：仅让普通沪深 A 股主板参与市值分位。
    # 因此科创板、创业板、北交所、新三板和 B 股均不会参与后续排序或交易。
    signal_sql = f"""
    WITH factor_base AS (
        SELECT
            date,
            instrument,
            cs_level1_name,
            total_market_cap,
            c_pct_rank(total_market_cap, ascending := true) AS mcap_pct,
            ({numerator_expr}) / NULLIF(({denominator_expr}), 0) AS exp_wgt_return_6m
        FROM cn_stock_prefactors
        WHERE date >= '{calc_start_date}'
          AND date <= '{END_DATE}'
          AND list_sector IN (1, 2, 3)
          AND is_risk_warning = 0
          AND suspended = 0
          AND list_days >= {MIN_LIST_DAYS}
          AND close IS NOT NULL
          AND turn IS NOT NULL
          AND total_market_cap IS NOT NULL
          AND cs_level1_name IS NOT NULL
          AND (
              instrument LIKE '600%.SH'
              OR instrument LIKE '601%.SH'
              OR instrument LIKE '603%.SH'
              OR instrument LIKE '605%.SH'
              OR instrument LIKE '000%.SZ'
              OR instrument LIKE '001%.SZ'
              OR instrument LIKE '002%.SZ'
              OR instrument LIKE '003%.SZ'
          )
    ),
    with_mcap_group AS (
        SELECT
            *,
            CASE
                WHEN mcap_pct IS NULL THEN NULL
                WHEN mcap_pct >= 1.0 THEN {MCAP_GROUP_COUNT}
                ELSE CAST(FLOOR(mcap_pct * {MCAP_GROUP_COUNT}) + 1 AS INTEGER)
            END AS mcap_group
        FROM factor_base
    ),
    universe AS (
        SELECT
            date,
            instrument,
            total_market_cap,
            mcap_group,
            exp_wgt_return_6m
        FROM with_mcap_group
        WHERE date IN ({rebalance_date_sql})
          AND mcap_group IN ({selected_group_sql})
          AND cs_level1_name NOT IN ({industry_sql})
          AND exp_wgt_return_6m IS NOT NULL
    ),
    ranked AS (
        SELECT
            date,
            instrument,
            mcap_group,
            exp_wgt_return_6m,
            COUNT(*) OVER (
                PARTITION BY date, mcap_group
            ) AS group_stock_count,
            ROW_NUMBER() OVER (
                PARTITION BY date, mcap_group
                ORDER BY exp_wgt_return_6m ASC, total_market_cap ASC, instrument ASC
            ) AS factor_rank_in_group
        FROM universe
    ),
    selected AS (
        SELECT
            *,
            CAST(CEIL(group_stock_count * {GROUP_SELECT_PCT:.12g}) AS INTEGER)
                AS group_select_num
        FROM ranked
    )
    SELECT
        date AS rebalance_signal_date,
        instrument,
        mcap_group,
        exp_wgt_return_6m,
        factor_rank_in_group
    FROM selected
    WHERE factor_rank_in_group <= group_select_num
    ORDER BY rebalance_signal_date, mcap_group, factor_rank_in_group, instrument
    """

    target_rebalance_df = dai.query(signal_sql).df()
    if target_rebalance_df.empty:
        raise ValueError("调仓候选池为空，请检查日期、过滤条件或因子字段。")

    target_rebalance_df["rebalance_signal_date"] = (
        pd.to_datetime(target_rebalance_df["rebalance_signal_date"])
        .dt.strftime("%Y-%m-%d")
    )
    target_rebalance_df = target_rebalance_df.sort_values(
        ["rebalance_signal_date", "mcap_group", "factor_rank_in_group", "instrument"]
    ).reset_index(drop=True)

    # 二次校验：任何不属于普通沪深 A 股主板的代码均不得进入收益交易信号。
    # 防御状态下的三只银行补偿股由独立分支加入，不受该收益候选池校验影响。
    ordinary_a_share_mask = target_rebalance_df["instrument"].astype(str).str.match(
        r"^(600|601|603|605)\d{3}\.SH$|^(000|001|002|003)\d{3}\.SZ$"
    )
    if not ordinary_a_share_mask.all():
        invalid_codes = target_rebalance_df.loc[
            ~ordinary_a_share_mask, "instrument"
        ].head(10).tolist()
        raise AssertionError(f"普通沪深 A 股主板白名单校验失败：{invalid_codes}")

    target_rebalance_df["instrument"] = target_rebalance_df["instrument"].astype(
        "category"
    )
    target_rebalance_df["mcap_group"] = target_rebalance_df["mcap_group"].astype(
        np.int16
    )
    target_rebalance_df["factor_rank_in_group"] = target_rebalance_df[
        "factor_rank_in_group"
    ].astype(np.int32)
    target_rebalance_df["exp_wgt_return_6m"] = target_rebalance_df[
        "exp_wgt_return_6m"
    ].astype(np.float32)

    trade_schedule_df = pd.DataFrame({"date": daily_trend_trade["date"].tolist()})
    effective_starts = np.array(
        rebalance_schedule_df["effective_start_date"].tolist(), dtype=object
    )
    active_index = np.searchsorted(
        effective_starts, trade_schedule_df["date"].values, side="right"
    ) - 1
    trade_schedule_df = trade_schedule_df.loc[active_index >= 0].copy()
    trade_schedule_df["rebalance_signal_date"] = np.array(
        rebalance_signal_dates, dtype=object
    )[active_index[active_index >= 0]]

    daily_target_df = trade_schedule_df.merge(
        target_rebalance_df, on="rebalance_signal_date", how="inner"
    ).merge(
        daily_trend_trade[
            ["date", "trend_signal_date", "risk_on", "trend_total_weight"]
        ],
        on="date",
        how="left",
    )
    if daily_target_df.empty:
        raise ValueError("每日目标持仓为空，请检查调仓日与执行日映射。")
    if daily_target_df[["risk_on", "trend_total_weight"]].isna().any().any():
        raise ValueError("每日目标持仓存在缺失的趋势信号。")

    # 原收益策略候选池：风险开启时合计 98%，防御时合计仅保留 10%。
    daily_target_df["stock_count"] = daily_target_df.groupby(
        "date", observed=True
    )["instrument"].transform("count")
    daily_target_df["weight"] = (
        daily_target_df["trend_total_weight"] / daily_target_df["stock_count"]
    )
    daily_target_df["compensation_total_weight"] = np.where(
        daily_target_df["risk_on"].eq(0),
        TARGET_TOTAL_WEIGHT - DEFENSIVE_TOTAL_WEIGHT,
        0.0,
    )
    daily_target_df["portfolio_target_weight"] = (
        daily_target_df["trend_total_weight"]
        + daily_target_df["compensation_total_weight"]
    )

    # 补偿资产独立于收益候选池构造：防御状态下使用从原收益信号中降下的
    # 88% 仓位，三只银行股等权；风险开启时不生成它们的目标持仓。
    defensive_meta_df = (
        daily_target_df.loc[
            daily_target_df["risk_on"].eq(0),
            [
                "date",
                "rebalance_signal_date",
                "trend_signal_date",
                "risk_on",
                "trend_total_weight",
                "compensation_total_weight",
                "portfolio_target_weight",
            ],
        ]
        .drop_duplicates("date")
        .reset_index(drop=True)
    )
    compensation_rows = []
    compensation_weight = (
        (TARGET_TOTAL_WEIGHT - DEFENSIVE_TOTAL_WEIGHT)
        / len(DEFENSIVE_COMPENSATION_INSTRUMENTS)
    )
    for row in defensive_meta_df.itertuples(index=False):
        for instrument in DEFENSIVE_COMPENSATION_INSTRUMENTS:
            compensation_rows.append(
                {
                    "date": row.date,
                    "instrument": instrument,
                    "weight": compensation_weight,
                    "rebalance_signal_date": row.rebalance_signal_date,
                    "trend_signal_date": row.trend_signal_date,
                    "risk_on": row.risk_on,
                    "trend_total_weight": row.trend_total_weight,
                    "compensation_total_weight": row.compensation_total_weight,
                    "portfolio_target_weight": row.portfolio_target_weight,
                }
            )

    main_signal_df = daily_target_df[
        [
            "date",
            "instrument",
            "weight",
            "rebalance_signal_date",
            "trend_signal_date",
            "risk_on",
            "trend_total_weight",
            "compensation_total_weight",
            "portfolio_target_weight",
        ]
    ].copy()
    compensation_signal_df = pd.DataFrame(
        compensation_rows,
        columns=main_signal_df.columns,
    )
    signal_df = pd.concat(
        [main_signal_df, compensation_signal_df],
        ignore_index=True,
    )
    signal_df["date"] = signal_df["date"].astype(str)
    signal_df["instrument"] = signal_df["instrument"].astype(str)
    signal_df["rebalance_signal_date"] = signal_df[
        "rebalance_signal_date"
    ].astype(str)
    signal_df["trend_signal_date"] = signal_df["trend_signal_date"].astype(str)
    signal_df["weight"] = signal_df["weight"].astype(np.float32)
    signal_df["risk_on"] = signal_df["risk_on"].astype(np.int8)
    signal_df["trend_total_weight"] = signal_df["trend_total_weight"].astype(
        np.float32
    )
    signal_df["compensation_total_weight"] = signal_df[
        "compensation_total_weight"
    ].astype(np.float32)
    signal_df["portfolio_target_weight"] = signal_df[
        "portfolio_target_weight"
    ].astype(np.float32)
    signal_df = signal_df.sort_values(["date", "instrument"]).reset_index(drop=True)

    if signal_df.empty:
        raise ValueError("signal_df 为空。")
    # 即使历史区间尚未出现防御日，也始终将补偿资产交给 BigTrader，
    # 以保证未来首次进入防御时可取得其当日行情并正常下单。
    all_instruments = sorted(
        set(signal_df["instrument"].unique().tolist())
        | set(DEFENSIVE_COMPENSATION_INSTRUMENTS)
    )
    if not all_instruments:
        raise ValueError("没有可回测标的。")

    effective_end_date = str(signal_df["date"].max())
    return signal_df, all_instruments, effective_end_date


# =====================================================
# 4. BigTrader 交易回调
# =====================================================

def _current_bar_values(data, instrument):
    try:
        row = data.current(instrument, ["open", "upper_limit", "lower_limit", "volume"])
        open_price = _safe_float(row["open"])
        upper_limit = _safe_float(row["upper_limit"])
        lower_limit = _safe_float(row["lower_limit"])
        volume = _safe_float(row["volume"], default=0.0)
    except Exception:
        return None

    if pd.isna(open_price) or pd.isna(upper_limit) or pd.isna(lower_limit):
        return None
    return {
        "open": open_price,
        "upper_limit": upper_limit,
        "lower_limit": lower_limit,
        "volume": volume,
    }


def _can_buy_at_current_open(data, instrument):
    bar = _current_bar_values(data, instrument)
    return bar is not None and bar["volume"] > 0 and bar["open"] < bar["upper_limit"] - EPS


def _can_sell_at_current_open(data, instrument):
    bar = _current_bar_values(data, instrument)
    return bar is not None and bar["volume"] > 0 and bar["open"] > bar["lower_limit"] + EPS


def _build_target_maps(signal_data):
    target_by_date = {}
    meta_by_date = {}
    meta_columns = [
        "rebalance_signal_date",
        "trend_signal_date",
        "risk_on",
        "trend_total_weight",
        "compensation_total_weight",
        "portfolio_target_weight",
    ]
    for date, group in signal_data.groupby("date", sort=False):
        target_by_date[date] = dict(
            zip(group["instrument"].values, group["weight"].astype(float).values)
        )
        first_row = group.iloc[0]
        meta_by_date[date] = {column: first_row[column] for column in meta_columns}
    return target_by_date, meta_by_date


def _set_current_bar_matching(context):
    if not USE_CURRENT_BAR_MATCHING:
        return
    try:
        vmatch_enum = getattr(bigtrader, "VMatchAt", None)
        if vmatch_enum is None:
            from bigtrader.constant import VMatchAt

            vmatch_enum = VMatchAt
        if hasattr(vmatch_enum, "CURRENT_BAR"):
            context.set_vmatch_at(vmatch_enum.CURRENT_BAR)
        elif hasattr(vmatch_enum, "CURRENT"):
            context.set_vmatch_at(vmatch_enum.CURRENT)
        else:
            raise AttributeError("未找到当前 Bar 撮合枚举。")
    except Exception as error:
        print("警告：未能设置当前 Bar 撮合模式：", repr(error))


def _load_desired_weight_map(context):
    try:
        stored = context.user_store.get("desired_weight_map", {})
        return dict(stored) if isinstance(stored, dict) else {}
    except Exception:
        return {}


def _save_desired_weight_map(context):
    try:
        context.user_store["desired_weight_map"] = context.desired_weight_map
    except Exception:
        pass


def _order_succeeded(result):
    return result is None or result == 0


def initialize(context):
    signal_data = context.data.copy()
    signal_data["date"] = signal_data["date"].astype(str)
    signal_data["instrument"] = signal_data["instrument"].astype(str)
    context.target_by_date, context.meta_by_date = _build_target_maps(signal_data)
    context.desired_weight_map = _load_desired_weight_map(context)
    context.set_commission(
        bigtrader.PerOrder(
            buy_cost=BUY_COST,
            sell_cost=SELL_COST,
            min_cost=MIN_COST,
        )
    )
    _set_current_bar_matching(context)


def handle_data(context, data):
    today = data.current_dt.strftime("%Y-%m-%d")
    target_map = context.target_by_date.get(today)
    if target_map is None:
        return

    if VERBOSE:
        meta = context.meta_by_date.get(today, {})
        print(
            f"{today} 执行：调仓信号={meta.get('rebalance_signal_date')}，"
            f"趋势信号={meta.get('trend_signal_date')}，"
            f"风险开关={meta.get('risk_on')}，"
            f"原信号仓位={float(meta.get('trend_total_weight', 0.0)):.2%}，"
            f"补偿仓位={float(meta.get('compensation_total_weight', 0.0)):.2%}，"
            f"组合目标仓位={float(meta.get('portfolio_target_weight', 0.0)):.2%}，"
            f"目标股票数={len(target_map)}"
        )

    holding_instruments = set(context.get_positions().keys())
    target_instruments = set(target_map.keys())

    # 不在目标池内的持仓，若交易条件允许则卖出。
    for instrument in sorted(holding_instruments - target_instruments):
        if not _can_sell_at_current_open(data, instrument):
            continue
        result = context.order_target_percent(instrument, 0)
        if _order_succeeded(result):
            context.desired_weight_map[instrument] = 0.0

    # 目标股票按目标权重调整；被涨跌停或停牌阻断时保留原仓位/现金，不再分配给其他股票。
    for instrument, target_weight in target_map.items():
        target_weight = float(target_weight)
        previous_weight = float(context.desired_weight_map.get(instrument, 0.0))
        weight_change = target_weight - previous_weight
        if abs(weight_change) < 1e-8:
            continue

        tradable = (
            _can_buy_at_current_open(data, instrument)
            if weight_change > 0
            else _can_sell_at_current_open(data, instrument)
        )
        if not tradable:
            continue

        result = context.order_target_percent(instrument, target_weight)
        if _order_succeeded(result):
            context.desired_weight_map[instrument] = target_weight

    _save_desired_weight_map(context)


# =====================================================
# 5. 运行回测
# =====================================================

signal_df, all_target_instruments, EFFECTIVE_END_DATE = build_signal_data()

print(f"查询结束日 END_DATE：{END_DATE}")
print(f"实际运行结束日 EFFECTIVE_END_DATE：{EFFECTIVE_END_DATE}")
print(f"初始资金：{CAPITAL_BASE:,.0f} 元")
print("收益候选池白名单：仅普通沪深 A 股主板；已排除科创板、创业板、北交所、新三板及 B 股")
print("防御补偿资产：601398.SH、601328.SH、601988.SH；仅在 MA60 防御状态下参与交易")
print(
    f"传入 BigTrader 的交易日数量：{signal_df['date'].nunique()}，"
    f"股票数量：{len(all_target_instruments)}"
)

performance = bigtrader.run(
    market=bigtrader.Market.CN_STOCK,
    frequency=bigtrader.Frequency.DAILY,
    start_date=START_DATE,
    end_date=EFFECTIVE_END_DATE,
    capital_base=CAPITAL_BASE,
    instruments=all_target_instruments,
    data=signal_df,
    initialize=initialize,
    handle_data=handle_data,
    benchmark=BENCHMARK,
    order_price_field_buy="open",
    order_price_field_sell="open",
    volume_limit=0.025,
)


查询结束日 END_DATE：2026-09-16
实际运行结束日 EFFECTIVE_END_DATE：2026-09-15
初始资金：100,000 元
收益候选池白名单：仅普通沪深 A 股主板；已排除科创板、创业板、北交所、新三板及 B 股
防御补偿资产：601398.SH、601328.SH、601988.SH；仅在 MA60 防御状态下参与交易
传入 BigTrader 的交易日数量：897，股票数量：144
[2026-09-16 10:03:55] [info     ] bigtrader init ..
[2026-09-16 10:03:55] [info     ] bigtrader.run start: market=cn_stock, frequency=1d, mode=backtest, account_type=STOCK, start_date=2023-01-01, end_date=2026-09-15
[2026-09-16 10:03:56] [info     ] bigtrader<backtest> init ..
[2026-09-16 10:03:56] [info     ] prepare data ..
[2026-09-16 10:03:56] [info     ] bar1d_df: (132945, 16)
[2026-09-16 10:03:56] [info     ] bigtrader use dividend data: (218, 8)
[2026-09-16 10:03:57] [info     ] bigtrader run ..


[2026-09-16 10:03:58] [info     ] bigtrader run done.
